In [1]:
# Setup
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import torch
from timesfm3 import TimesFM3Forecaster

device = "cuda" if torch.cuda.is_available() else "cpu"
pipeline = TimesFM3Forecaster.from_pretrained("google/timesfm-3.0-pytorch", device=device)

MODEL_LABEL = "TimesFM 3.0"
MAX_CONTEXT = pipeline.global_context
MAX_HORIZON = 1024
print(f"Loaded {MODEL_LABEL} on {device}")

Loaded TimesFM 3.0 on cuda


In [2]:
# Load data
import json

DATA_FILE = "tep_sp_spaced_6min_constdist_noisy.h5"


def read_names(ds):
    return [n.decode() if isinstance(n, bytes) else str(n) for n in ds.attrs["names"]]


groups = {}
with h5py.File(DATA_FILE, "r") as f:
    time_h = f["t"][()] / 3600
    for key in ["xmeas", "xmv", "dist", "sp"]:
        groups[key] = (f[key][()], read_names(f[key]))
    if "y" in f:   # true valve positions
        groups["valve"] = (f["y"][:, 38:50], read_names(f["y"])[38:50])
    config = json.loads(f.attrs["config"])
SAMPLE_MIN = config["record_every"] / 60   # sampling interval (min)

nominal = {int(k): v for k, v in config["setpoints"].items()}
CHANGED = sorted({k for k, _, _ in config["setpoint_schedule"]})   # set points changed in this run
t_test = config.get("t_test")                                    # start of the test changes (h)

sp = groups["sp"][0]
off = {k: np.abs(sp[:, k - 1] - nominal[k]) > 1e-9 for k in CHANGED}
sp_change = np.zeros(len(time_h), dtype=int)
for k in reversed(CHANGED):
    sp_change[off[k]] = k

data = {"time_h": time_h, "sp_change": sp_change}
COLUMNS, LABELS = {}, {}
for group, (values, labels) in groups.items():
    COLUMNS[group] = [f"{group}_{i}" for i in range(1, values.shape[1] + 1)]
    for col, v, label in zip(COLUMNS[group], values.T, labels):
        data[col] = v
        LABELS[col] = label

df = pd.DataFrame(data)
df.insert(0, "item_id", "tep")
df.insert(1, "timestamp", pd.date_range("2026-01-01", periods=len(df), freq=f"{config['record_every']}s"))

TARGET_COLUMNS = COLUMNS["xmeas"] + COLUMNS["xmv"]
print(f"{len(df)} samples ({time_h[-1]:.0f} h), {len(TARGET_COLUMNS)} targets, covariate groups: "
      + ", ".join(f"{g} ({len(COLUMNS[g])})" for g in COLUMNS if g not in ["xmeas", "xmv"]))
print(f"changed set points: {CHANGED}, test part from {t_test} h")
df.head()

6798 samples (680 h), 53 targets, covariate groups: dist (12), sp (20), valve (12)
changed set points: [5, 7, 8, 11, 13, 14, 15, 17, 18, 19, 20], test part from 580.6 h


,item_id,timestamp,time_h,sp_change,xmeas_1,xmeas_2,xmeas_3,xmeas_4,xmeas_5,xmeas_6,...,valve_3,valve_4,valve_5,valve_6,valve_7,valve_8,valve_9,valve_10,valve_11,valve_12
0,tep,2026-01-01 00:00:00,0.1,0,0.250570,3646.911201,4529.162679,9.362211,26.955963,42.373414,...,24.683121,61.479204,22.363783,40.094634,38.425616,46.451781,47.454458,41.014715,17.044840,50.0
1,tep,2026-01-01 00:06:00,0.2,0,0.251641,3685.782817,4532.305731,9.392836,27.191772,42.304758,...,24.669825,61.225171,22.292722,39.989795,38.837808,46.079812,47.453757,40.959467,19.485093,50.0
2,tep,2026-01-01 00:12:00,0.3,0,0.224988,3649.653552,4485.108841,9.373259,26.671660,42.429623,...,21.911699,61.458982,22.123348,40.747079,37.087973,47.818568,47.557016,40.224623,16.873591,50.0
3,tep,2026-01-01 00:18:00,0.4,0,0.218616,3639.422517,4472.533021,9.491735,26.897842,42.106417,...,21.639985,61.548652,22.355574,40.738591,34.571177,46.714696,47.677321,40.833625,17.571302,50.0
4,tep,2026-01-01 00:24:00,0.5,0,0.259312,3666.070767,4583.076516,9.291715,27.026806,42.234145,...,25.624424,61.177210,21.877342,42.039533,38.116623,48.398896,47.830919,40.202709,18.061803,50.0


In [3]:
# Test events and forecast windows
events = []   # (start index, end index, set point, new value)
for k in CHANGED:
    on = off[k]
    for i in range(len(on)):
        if on[i] and (i == 0 or not on[i - 1]):
            start = i
        if on[i] and (i == len(on) - 1 or not on[i + 1]):
            events.append((start, i, k, sp[start, k - 1]))
events.sort()

BUFFER = 20   # nominal samples before the event at the start of each forecast
# Samples where a disturbance starts to ramp to a new operating point, none for most datasets
ramp_starts = sorted({int(np.searchsorted(time_h, t0)) for _, t0, _, _ in config.get("dist_schedule") or []})

# One forecast per test event, from BUFFER samples before the event up to the start of the next event or of
# the next disturbance ramp
rows = []
for n, (s, e, k, v) in enumerate(events):
    if t_test is None or time_h[s] < t_test:
        continue
    split = max(s - BUFFER, 0)
    end = events[n + 1][0] if n + 1 < len(events) else len(df)
    end = min([end] + [r for r in ramp_starts if r > e])
    if end <= e:
        raise ValueError(f"event {n} is still running when the next event starts")
    horizon = min(end - split, MAX_HORIZON)
    rows.append({
        "event": n,
        "set point": f"SP {k} {LABELS[f'sp_{k}']}",
        "change (%)": 100 * (v - nominal[k]) / nominal[k],
        "start (h)": time_h[split],
        "horizon (h)": horizon * SAMPLE_MIN / 60,
        "split": split,
        "horizon": horizon,
    })
windows = pd.DataFrame(rows).set_index("event")

EXCLUDED = []   # test events left out of the scores
windows = windows.drop(EXCLUDED)
print(f"{len(windows)} test events")
windows.round(2)

11 test events


,set point,change (%),start (h),horizon (h),split,horizon
event,,,,,,
66,SP 11 Stripper underflow (m3/h),1.74,578.6,10.9,5785,109
67,SP 5 Recycle flow (kscmh),-1.86,587.5,8.9,5874,89
68,SP 14 Reactor feed D (mol%),2.91,594.4,12.3,5943,123
69,SP 20 Product E (mol%),-7.18,604.7,12.8,6046,128
70,SP 18 Reactor temperature (C),0.62,615.5,6.5,6154,65
71,SP 15 Reactor feed E (mol%),-3.99,620.0,12.1,6199,121
72,SP 13 Reactor feed A (mol%),-1.86,630.1,12.8,6300,128
73,SP 17 Reactor level (%),-1.67,640.9,10.0,6408,100
74,SP 19 Purge B (mol%),7.23,648.9,12.8,6488,128


In [4]:
# Covariates and scored variables
USE_DIST = True                  # disturbance signals (12)
USE_SP = True                    # changed set points (11)
USE_VALVE = False                # true valve positions (12)
DROPPED = []
PAST_ONLY = []
COVARIATES_KNOWN_FUTURE = True

ALL_COVARIATES = []
for group, use in [("dist", USE_DIST), ("sp", USE_SP), ("valve", USE_VALVE)]:
    if use and group in COLUMNS:
        ALL_COVARIATES += [f"sp_{k}" for k in CHANGED] if group == "sp" else COLUMNS[group]
ALL_COVARIATES = [c for c in ALL_COVARIATES if c not in DROPPED]


def covariates(window_df):
    """Covariates for one forecast window, leaving out those that are constant over it."""
    used = [c for c in ALL_COVARIATES if window_df[c].nunique() > 1]
    future = [c for c in used if COVARIATES_KNOWN_FUTURE and c not in PAST_ONLY]
    past_only = [c for c in used if c not in future]
    return used, future, past_only


EXCLUDED_VARIABLES = ["xmeas_37"]   # Product D, about 0.017 mol%: near zero, so its percentage error explodes
SCORED = [c for c in COLUMNS["xmeas"] if c not in EXCLUDED_VARIABLES]   # measurements only, no valves
SCALE_FILE = "tep_sp_spaced_6min_noisy.h5"   # dataset whose standard deviation scales the errors, None for this dataset
if SCALE_FILE is None:
    SCALE = df[SCORED].std()    # standard deviation over the whole dataset
else:   # same scale for several datasets, so their scaled scores are comparable
    with h5py.File(SCALE_FILE, "r") as f:
        SCALE = pd.DataFrame(f["xmeas"][()], columns=COLUMNS["xmeas"])[SCORED].std()
    print(f"errors scaled by the standard deviation of {SCALE_FILE}")
print(f"scored variables ({len(SCORED)}): {', '.join(SCORED)}")
print(f"covariates ({len(ALL_COVARIATES)}): {', '.join(ALL_COVARIATES) or '-'}")

errors scaled by the standard deviation of tep_sp_spaced_6min_noisy.h5
scored variables (40): xmeas_1, xmeas_2, xmeas_3, xmeas_4, xmeas_5, xmeas_6, xmeas_7, xmeas_8, xmeas_9, xmeas_10, xmeas_11, xmeas_12, xmeas_13, xmeas_14, xmeas_15, xmeas_16, xmeas_17, xmeas_18, xmeas_19, xmeas_20, xmeas_21, xmeas_22, xmeas_23, xmeas_24, xmeas_25, xmeas_26, xmeas_27, xmeas_28, xmeas_29, xmeas_30, xmeas_31, xmeas_32, xmeas_33, xmeas_34, xmeas_35, xmeas_36, xmeas_38, xmeas_39, xmeas_40, xmeas_41
covariates (23): dist_1, dist_2, dist_3, dist_4, dist_5, dist_6, dist_7, dist_8, dist_9, dist_10, dist_11, dist_12, sp_5, sp_7, sp_8, sp_11, sp_13, sp_14, sp_15, sp_17, sp_18, sp_19, sp_20


In [5]:
# Metrics
QUANTILES = [0.1, 0.5, 0.9]
ALPHA = round(1 - (QUANTILES[-1] - QUANTILES[0]), 6)   # 0.2 for the 10-90% interval
METRICS = ["scaled MAE", "MAPE (%)", "coverage", "scaled Winkler"]


def winkler(actual, lower, upper, alpha):
    """Winkler interval score at every step."""
    width = upper - lower
    below = width + 2 / alpha * (lower - actual)
    above = width + 2 / alpha * (actual - upper)
    return np.where(actual < lower, below, np.where(actual > upper, above, width))


def score_event(forecast, truth_df):
    """Metrics per scored variable for one forecast, forecast[col] = (point forecast, lower, upper)."""
    rows = {}
    for col in SCORED:
        pred, lower, upper = forecast[col]
        actual = truth_df[col].to_numpy()
        rows[col] = {
            "scaled MAE": np.mean(np.abs(pred - actual)) / SCALE[col],
            "MAPE (%)": 100 * np.mean(np.abs(actual - pred) / np.abs(actual)),
            "coverage": np.mean((actual >= lower) & (actual <= upper)),
            "scaled Winkler": np.mean(winkler(actual, lower, upper, ALPHA)) / SCALE[col],
        }
    return pd.DataFrame(rows).T

In [6]:
# TimesFM 3.0 forecast of one window
VARIATES_PER_CALL = None

q_index = {q: pipeline.config.quantiles.index(q) for q in QUANTILES}


def run_forecast(context_df, truth_df, used, future, past_only):
    horizon = len(truth_df)
    if VARIATES_PER_CALL is None:
        targets_per_call = len(TARGET_COLUMNS)
    else:
        targets_per_call = VARIATES_PER_CALL - len(used)
        if targets_per_call < 1:
            raise ValueError(f"{len(used)} covariates leave no room for targets in {VARIATES_PER_CALL} series")

    past_only_cov = None
    if past_only:
        past_only_cov = context_df[past_only].to_numpy(dtype=np.float32).T                 # (covariates, context)
    past_future_cov = None
    if future:
        window_df = pd.concat([context_df, truth_df])
        past_future_cov = window_df[future].to_numpy(dtype=np.float32).T                   # (covariates, context + horizon)

    median, quantiles = [], []
    for i in range(0, len(TARGET_COLUMNS), targets_per_call):
        chunk = TARGET_COLUMNS[i:i + targets_per_call]
        out = pipeline.predict(
            context_df[chunk].to_numpy(dtype=np.float32).T,                                    # (targets, context)
            horizon=horizon,
            past_only_covariates=past_only_cov,
            past_future_covariates=past_future_cov,
            return_quantiles=True,
        )
        median.append(out.forecast[:, :horizon])
        quantiles.append(out.quantiles[:, :horizon])
    median = np.concatenate(median)          # (targets, horizon)
    quantiles = np.concatenate(quantiles)    # (targets, horizon, 9) deciles 0.1 .. 0.9

    forecast = {}
    for v, col in enumerate(TARGET_COLUMNS):
        forecast[col] = (median[v], quantiles[v, :, q_index[QUANTILES[0]]], quantiles[v, :, q_index[QUANTILES[-1]]])
    return forecast

In [7]:
# Forecast every test event
per_variable = {}   # event -> metrics per scored variable
for n, w in windows.iterrows():
    split, horizon = int(w["split"]), int(w["horizon"])
    context_used = min(split, MAX_CONTEXT)
    context_df = df.iloc[split - context_used:split]
    truth_df = df.iloc[split:split + horizon]
    used, future, past_only = covariates(pd.concat([context_df, truth_df]))

    forecast = run_forecast(context_df, truth_df, used, future, past_only)
    per_variable[n] = score_event(forecast, truth_df)
    means = per_variable[n].mean()
    print(f"event {n:2d} {w['set point']}: " + ", ".join(f"{m} {means[m]:.3f}" for m in METRICS))

event 66 SP 11 Stripper underflow (m3/h): scaled MAE 0.507, MAPE (%) 2.170, coverage 0.846, scaled Winkler 2.268


event 67 SP 5 Recycle flow (kscmh): scaled MAE 0.529, MAPE (%) 2.143, coverage 0.846, scaled Winkler 2.350


event 68 SP 14 Reactor feed D (mol%): scaled MAE 0.412, MAPE (%) 1.776, coverage 0.909, scaled Winkler 2.128


event 69 SP 20 Product E (mol%): scaled MAE 0.429, MAPE (%) 1.765, coverage 0.901, scaled Winkler 2.095


event 70 SP 18 Reactor temperature (C): scaled MAE 0.562, MAPE (%) 2.419, coverage 0.812, scaled Winkler 2.325


event 71 SP 15 Reactor feed E (mol%): scaled MAE 0.530, MAPE (%) 2.449, coverage 0.871, scaled Winkler 2.346


event 72 SP 13 Reactor feed A (mol%): scaled MAE 0.457, MAPE (%) 2.057, coverage 0.875, scaled Winkler 2.084


event 73 SP 17 Reactor level (%): scaled MAE 0.595, MAPE (%) 2.525, coverage 0.812, scaled Winkler 2.498


event 74 SP 19 Purge B (mol%): scaled MAE 0.551, MAPE (%) 2.422, coverage 0.826, scaled Winkler 2.386


event 75 SP 7 Separator level (%): scaled MAE 0.535, MAPE (%) 2.249, coverage 0.853, scaled Winkler 2.310


event 76 SP 8 Stripper level (%): scaled MAE 0.557, MAPE (%) 2.475, coverage 0.790, scaled Winkler 2.409


In [8]:
# Scores per event, averaged over the scored variables
event_scores = pd.DataFrame({n: s.mean() for n, s in per_variable.items()}).T[METRICS]
event_table = windows[["set point", "change (%)", "start (h)", "horizon (h)"]].join(event_scores)
print(f"{MODEL_LABEL}: mean over {len(SCORED)} measurements per event")
event_table.round(3)

TimesFM 3.0: mean over 40 measurements per event


,set point,change (%),start (h),horizon (h),scaled MAE,MAPE (%),coverage,scaled Winkler
event,,,,,,,,
66,SP 11 Stripper underflow (m3/h),1.743,578.6,10.9,0.507,2.170,0.846,2.268
67,SP 5 Recycle flow (kscmh),-1.859,587.5,8.9,0.529,2.143,0.846,2.350
68,SP 14 Reactor feed D (mol%),2.906,594.4,12.3,0.412,1.776,0.909,2.128
69,SP 20 Product E (mol%),-7.180,604.7,12.8,0.429,1.765,0.901,2.095
70,SP 18 Reactor temperature (C),0.623,615.5,6.5,0.562,2.419,0.812,2.325
71,SP 15 Reactor feed E (mol%),-3.994,620.0,12.1,0.530,2.449,0.871,2.346
72,SP 13 Reactor feed A (mol%),-1.864,630.1,12.8,0.457,2.057,0.875,2.084
73,SP 17 Reactor level (%),-1.667,640.9,10.0,0.595,2.525,0.812,2.498
74,SP 19 Purge B (mol%),7.234,648.9,12.8,0.551,2.422,0.826,2.386


In [9]:
# Average over all test events
final = event_scores.agg(["mean", "std"])   # mean and standard deviation over the test events
print(f"{MODEL_LABEL}: mean over {len(event_scores)} test events")
final.round(3)

TimesFM 3.0: mean over 11 test events


,scaled MAE,MAPE (%),coverage,scaled Winkler
mean,0.515,2.223,0.849,2.291
std,0.058,0.270,0.038,0.135


In [10]:
# Save scores
import os

RESULTS_FILE = "Results/Loop_setpointforecast_timesfm3_constdist_futuredist.h5"

os.makedirs("Results", exist_ok=True)
with h5py.File(RESULTS_FILE, "w") as f:
    f.attrs["model"] = MODEL_LABEL
    f.attrs["settings"] = json.dumps({
        "data_file": DATA_FILE,
        "buffer": BUFFER,
        "quantiles": QUANTILES,
        "scored": SCORED,
        "covariates": ALL_COVARIATES,
        "past_only": PAST_ONLY,
        "excluded_events": EXCLUDED,
        "excluded_variables": EXCLUDED_VARIABLES,
        "scale_file": SCALE_FILE or DATA_FILE,
    })
    f.create_dataset("event", data=event_scores.index.to_numpy())
    f.create_dataset("split", data=windows.loc[event_scores.index, "split"].to_numpy())
    f.create_dataset("horizon", data=windows.loc[event_scores.index, "horizon"].to_numpy())
    for name in METRICS:
        f.create_dataset(f"per_event/{name}", data=event_scores[name].to_numpy())
        f.create_dataset(f"average/{name}", data=final.loc["mean", name])
        f.create_dataset(f"std/{name}", data=final.loc["std", name])   # over the test events
        f.create_dataset(f"per_variable/{name}",   # events x scored variables
                         data=np.array([per_variable[n].loc[SCORED, name].to_numpy() for n in event_scores.index]))
print(f"Saved {MODEL_LABEL} scores to {RESULTS_FILE}")

Saved TimesFM 3.0 scores to Results/Loop_setpointforecast_timesfm3_constdist_futuredist.h5
